# SmartCart Machine Learning Pipeline

This notebook trains models for return-risk classification and customer-value prediction using the cleaned SmartCart dataset.

In [1]:
import pandas as pd
from pathlib import Path

file_name = 'cleaned_customer_shopping_data.csv'
matches = list(Path.cwd().parent.rglob(file_name))

if not matches:
    raise FileNotFoundError(
        f'{file_name} was not found under {Path.cwd()}. '
        'Run the ETL pipeline first.'
    )

df_clean = pd.read_csv(matches[0])
print(f'Dataframe loaded successfully: {df_clean.shape}')

Dataframe loaded successfully: (10000, 32)


## 📉 Goal 1: Return-Risk Classification

Logistic Regression is compared with Random Forest. F1 score is prioritised because returned orders are the minority class and both precision and recall matter.

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, f1_score, roc_auc_score

features_for_returns = [
    'Age', 'Gender', 'Category', 'Brand', 'Size', 'Color',
    'Discount (%)', 'Purchase Amount (₹)'
]
X = pd.get_dummies(df_clean[features_for_returns], drop_first=True)
y = df_clean['Return Status'].astype(str).str.strip().str.lower().eq('returned').astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

return_model = RandomForestClassifier(
    class_weight='balanced', random_state=42
)
return_model.fit(X_train, y_train)

logistic_model = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(
        class_weight='balanced', max_iter=1000, random_state=42
    ))
])
logistic_model.fit(X_train, y_train)

models = {
    'Random Forest': return_model,
    'Logistic Regression': logistic_model
}
results = {}

for name, model in models.items():
    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1]
    results[name] = {
        'F1 score': f1_score(y_test, predictions),
        'ROC-AUC': roc_auc_score(y_test, probabilities)
    }
    print(f'--- {name} ---')
    print(classification_report(y_test, predictions, zero_division=0))

comparison = pd.DataFrame(results).T.sort_values('F1 score', ascending=False)
display(comparison)
best_model = comparison.index[0]
print(f'Preferred model based on F1 score: {best_model}')

--- Random Forest ---
              precision    recall  f1-score   support

           0       0.82      1.00      0.90      1640
           1       0.00      0.00      0.00       360

    accuracy                           0.82      2000
   macro avg       0.41      0.50      0.45      2000
weighted avg       0.67      0.82      0.74      2000

--- Logistic Regression ---
              precision    recall  f1-score   support

           0       0.84      0.46      0.59      1640
           1       0.19      0.60      0.29       360

    accuracy                           0.48      2000
   macro avg       0.52      0.53      0.44      2000
weighted avg       0.72      0.48      0.54      2000



,F1 score,ROC-AUC
Logistic Regression,0.293115,0.534326
Random Forest,0.000000,0.503863


Preferred model based on F1 score: Logistic Regression


### Goal 1 Explanation and Conclusion

The target variable is whether a transaction was returned. Categorical predictors such as gender, category, brand, size, and colour are converted into numeric indicator columns using one-hot encoding. The data is split into training and test sets using stratification so that both sets contain a similar proportion of returned orders.

Returns are the minority class, so both models use `class_weight='balanced'`. Logistic Regression is placed in a pipeline with `StandardScaler` because scaling helps models that depend on feature magnitude. Random Forest is also tested because it can learn non-linear relationships and interactions without scaling.

The **F1 score** is the main comparison metric because it balances precision and recall for the returned-order class. ROC-AUC is included to measure how well each model ranks return risk across different probability thresholds.

**Conclusion:** The preferred model is the one identified in the output as having the highest F1 score. It should be used to flag potentially risky orders, while the classification report should be reviewed to ensure that the balance between missed returns and unnecessary alerts is acceptable for the business.

## 💎 Goal 2: Customer-Value Regression

A Random Forest Regressor estimates total customer spend from loyalty and engagement features.

In [3]:
from sklearn.ensemble import RandomForestRegressor

customer_profile = df_clean.groupby('Customer ID').agg({
    'Purchase Amount (₹)': 'sum',
    'Previous Purchases': 'max',
    'Frequency of Purchases': 'first',
    'Subscription Status': 'first',
    'Review Rating': 'mean'
}).reset_index()

X_cust = pd.get_dummies(
    customer_profile[[
        'Previous Purchases', 'Frequency of Purchases',
        'Subscription Status', 'Review Rating'
    ]],
    drop_first=True
)
y_cust = customer_profile['Purchase Amount (₹)']

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_cust, y_cust, test_size=0.2, random_state=42
)
value_model = RandomForestRegressor(random_state=42)
value_model.fit(X_train_c, y_train_c)
print(f'Model R² Score: {value_model.score(X_test_c, y_test_c):.2f}')

Model R² Score: 0.63


### Goal 2 Explanation and Conclusion

This goal estimates customer value by grouping transactions by `Customer ID` and summing each customer’s purchase amounts. Loyalty and engagement variables—previous purchases, purchase frequency, subscription status, and average review rating—are used as predictors. Categorical predictors are converted into numeric columns with one-hot encoding before training the Random Forest Regressor.

The data is divided into training and test sets. The model learns patterns from the training customers and is evaluated on customers it did not see during training. The displayed **$R^2$ score** indicates the proportion of variation in total customer spend explained by the predictors: values closer to 1 indicate stronger predictive performance, while values near 0 indicate limited predictive value.

**Conclusion:** Use the displayed test-set $R^2$ score to judge whether the model is useful for identifying high-value customers. A strong positive score would support targeted loyalty offers and customer segmentation. A low or negative score would indicate that additional predictors—such as transaction count, recency, average order value, channel, discount usage, or delivery experience—are needed before using the model for business decisions.

## Interactive stakeholder prototype

The following prototype adds user-controlled filters, dynamic Plotly charts, and prediction forms. Stakeholders can explore the data and test example order or customer profiles without changing the underlying analysis.

The recommended production tool is a **Streamlit dashboard with Plotly**: it provides a browser-based interface, reusable filters, interactive charts, and model prediction forms.

In [4]:
import plotly.express as px
import ipywidgets as widgets
from IPython.display import clear_output, display


def values_for(column):
    return ['All'] + sorted(df_clean[column].dropna().astype(str).unique().tolist())


category_filter = widgets.Dropdown(options=values_for('Category'), description='Category:')
channel_filter = widgets.Dropdown(options=values_for('Online/Offline'), description='Channel:')
festival_filter = widgets.Dropdown(options=values_for('Festival/Sale'), description='Festival:')
gender_filter = widgets.Dropdown(options=values_for('Gender'), description='Gender:')
discount_filter = widgets.IntRangeSlider(
    value=[int(df_clean['Discount (%)'].min()), int(df_clean['Discount (%)'].max())],
    min=int(df_clean['Discount (%)'].min()),
    max=int(df_clean['Discount (%)'].max()),
    description='Discount:',
)


def update_dashboard(category, channel, festival, gender, discount):
    filtered = df_clean[df_clean['Discount (%)'].between(discount[0], discount[1])].copy()
    selections = {
        'Category': category,
        'Online/Offline': channel,
        'Festival/Sale': festival,
        'Gender': gender,
    }
    for column, selection in selections.items():
        if selection != 'All':
            filtered = filtered[filtered[column].astype(str) == selection]

    with dashboard_output:
        clear_output(wait=True)
        if filtered.empty:
            print('No transactions match the selected filters.')
            return

        print(f'Transactions displayed: {len(filtered):,}')
        category_summary = filtered.groupby('Category', as_index=False).agg(
            average_purchase=('Purchase Amount (₹)', 'mean'),
            average_quantity=('Quantity', 'mean'),
            return_rate=('Is_Returned', 'mean'),
        )
        display(px.bar(
            category_summary,
            x='Category',
            y='average_purchase',
            color='Category',
            title='Average purchase amount by category',
        ))
        display(px.scatter(
            filtered,
            x='Discount (%)',
            y='Purchase Amount (₹)',
            color='Category',
            size='Quantity',
            hover_data=['Review Rating', 'Return Status'],
            title='Discount versus purchase amount',
        ))
        display(px.histogram(
            filtered,
            x='Review Rating',
            color='Return Status',
            barmode='group',
            title='Review rating by return status',
        ))


dashboard_output = widgets.Output()
dashboard_controls = widgets.VBox([
    widgets.HBox([category_filter, channel_filter]),
    widgets.HBox([festival_filter, gender_filter]),
    discount_filter,
])
dashboard_interaction = widgets.interactive_output(
    update_dashboard,
    {
        'category': category_filter,
        'channel': channel_filter,
        'festival': festival_filter,
        'gender': gender_filter,
        'discount': discount_filter,
    },
)
display(dashboard_controls, dashboard_output, dashboard_interaction)

Output()

Output()

In [10]:
def unique_values(column):
    return sorted(df_clean[column].dropna().astype(str).unique().tolist())


return_inputs = {
    'Age': widgets.IntSlider(
        value=int(df_clean['Age'].median()),
        min=int(df_clean['Age'].min()),
        max=int(df_clean['Age'].max()),
        description='Age:',
    ),
    'Gender': widgets.Dropdown(options=unique_values('Gender'), description='Gender:'),
    'Category': widgets.Dropdown(options=unique_values('Category'), description='Category:'),
    'Brand': widgets.Dropdown(options=unique_values('Brand'), description='Brand:'),
    'Size': widgets.Dropdown(options=unique_values('Size'), description='Size:'),
    'Color': widgets.Dropdown(options=unique_values('Color'), description='Colour:'),
    'Discount (%)': widgets.IntSlider(
        value=int(df_clean['Discount (%)'].median()),
        min=int(df_clean['Discount (%)'].min()),
        max=int(df_clean['Discount (%)'].max()),
        description='Discount:',
    ),
    'Purchase Amount (₹)': widgets.FloatSlider(
        value=float(df_clean['Purchase Amount (₹)'].median()),
        min=float(df_clean['Purchase Amount (₹)'].min()),
        max=float(df_clean['Purchase Amount (₹)'].max()),
        step=1,
        description='Amount:',
    ),
}
return_prediction_output = widgets.Output()


def predict_return_risk(**values):
    input_frame = pd.DataFrame([values])
    input_frame = pd.get_dummies(input_frame, drop_first=True).reindex(
        columns=X.columns, fill_value=0
    )
    probability = models[best_model].predict_proba(input_frame)[0, 1]
    with return_prediction_output:
        clear_output(wait=True)
        print(f'Model used: {best_model}')
        print(f'Estimated return probability: {probability:.1%}')
        print('Risk category:', 'High risk' if probability >= 0.5 else 'Low risk')


return_prediction_interaction = widgets.interactive_output(
    predict_return_risk, return_inputs
)
display(widgets.VBox(list(return_inputs.values())), return_prediction_output, return_prediction_interaction)

Output()

Output()

In [11]:
value_inputs = {
    'Previous Purchases': widgets.IntSlider(
        value=int(df_clean['Previous Purchases'].median()),
        min=int(df_clean['Previous Purchases'].min()),
        max=int(df_clean['Previous Purchases'].max()),
        description='Previous:',
    ),
    'Frequency of Purchases': widgets.Dropdown(
        options=unique_values('Frequency of Purchases'), description='Frequency:'
    ),
    'Subscription Status': widgets.Dropdown(
        options=unique_values('Subscription Status'), description='Subscription:'
    ),
    'Review Rating': widgets.FloatSlider(
        value=float(df_clean['Review Rating'].median()),
        min=float(df_clean['Review Rating'].min()),
        max=float(df_clean['Review Rating'].max()),
        step=0.1,
        description='Rating:',
    ),
}
value_prediction_output = widgets.Output()


def predict_customer_value(**values):
    input_frame = pd.DataFrame([values])
    input_frame = pd.get_dummies(input_frame, drop_first=True).reindex(
        columns=X_cust.columns, fill_value=0
    )
    estimate = value_model.predict(input_frame)[0]
    with value_prediction_output:
        clear_output(wait=True)
        print(f'Estimated total customer value: ₹{estimate:,.2f}')


value_prediction_interaction = widgets.interactive_output(
    predict_customer_value, value_inputs
)
display(widgets.VBox(list(value_inputs.values())), value_prediction_output, value_prediction_interaction)

Output()

Output()